# COREY selective_scan_fn Benchmark on TPU/GPU (Colab Ready)

本 notebook 支持在 Google Colab 上直接运行 COREY selective_scan_fn/benchmark，自动检测 TPU/GPU/CPU，输出 mean/std latency，结果可用于论文表格和对比实验。

In [ ]:
# 安装 PyTorch XLA（TPU 环境自动安装）
import os
if 'COLAB_TPU_ADDR' in os.environ:
    VERSION = '2.1'
    !pip install torch==2.1.0 torch_xla==2.1 -f https://storage.googleapis.com/libtorch-nightly/index.html
else:
    !pip install torch
        

In [ ]:
import os
import time
import torch
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    print('Using TPU:', device)
    is_tpu = True
except ImportError:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Using device:', device)
    is_tpu = False

# 参数
batch = 1
seq_len = 4096
dim = 1024
d_state = 16
chunk_size = 512
dtype = torch.float16
repeat = 30

# 随机输入
u = torch.randn(batch, dim, seq_len, device=device, dtype=dtype)
delta = torch.rand(batch, dim, seq_len, device=device, dtype=dtype)
A = torch.randn(dim, d_state, device=device, dtype=torch.float32)
B = torch.randn(dim, d_state, device=device, dtype=torch.float32)
C = torch.randn(dim, d_state, device=device, dtype=torch.float32)
D = torch.randn(dim, device=device, dtype=torch.float32)

# Dummy selective_scan_fn（如有正式实现请替换）
def selective_scan_fn(u, delta, A, B, C, D=None):
    return u + delta + A.sum() + B.sum() + C.sum() + (D.sum() if D is not None else 0)

# Warmup
for _ in range(3):
    _ = selective_scan_fn(u, delta, A, B, C, D)
    if is_tpu:
        xm.mark_step()
    elif device.type == 'cuda':
        torch.cuda.synchronize()

# Benchmark
latencies = []
for i in range(repeat):
    t0 = time.time()
    _ = selective_scan_fn(u, delta, A, B, C, D)
    if is_tpu:
        xm.mark_step()
    elif device.type == 'cuda':
        torch.cuda.synchronize()
    t1 = time.time()
    latencies.append((t1 - t0) * 1000)

mean_latency = sum(latencies) / len(latencies)
std_latency = (sum((x - mean_latency) ** 2 for x in latencies) / len(latencies)) ** 0.5
print(f'Mean latency: {mean_latency:.3f} ms, Std: {std_latency:.3f} ms')
# 可选：保存结果
import json
result = {
    'device': str(device),
    'mean_latency_ms': mean_latency,
    'std_latency_ms': std_latency,
    'latencies_ms': latencies,
}
with open('corey_tpu_benchmark_result.json', 'w') as f:
    json.dump(result, f, indent=2)
print('Saved to corey_tpu_benchmark_result.json')

In [ ]:
# 结果读取与健壮性检查（避免 JSON 解析错误）
import os
import json
if os.path.exists('corey_tpu_benchmark_result.json') and os.path.getsize('corey_tpu_benchmark_result.json') > 0:
    with open('corey_tpu_benchmark_result.json') as f:
        try:
            result = json.load(f)
            print('Benchmark result:', result)
        except json.JSONDecodeError as e:
            print('JSON decode error:', e)
else:
    print('Result file not found or is empty.')
